In [74]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [75]:
# 1. Charger les données

def load_words(path="names.txt"):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()

w = load_words()
words = w[:30]

In [76]:
# 2. Vocabulaire

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

In [77]:
vocab_size

22

In [78]:
# 3. Dataset (contexte = 3)

block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [79]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 11],
        [ 5, 11, 11],
        [11, 11,  1],
        [ 0,  0,  0],
        [ 0,  0, 13],
        [ 0, 13, 10],
        [13, 10,  9],
        [10,  9, 19],
        [ 9, 19,  9],
        [19,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 19],
        [ 1, 19,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 16],
        [ 9, 16,  1],
        [16,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 10],
        [ 5, 10, 10],
        [10, 10,  1],
        [ 0,  0,  0],
        [ 0,  0, 16],
        [ 0, 16, 13],
        [16, 13, 14],
        [13, 14,  8],
        [14,  8,  9],
        [ 8,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  3],
        [ 0,  3,  8],
        [ 3,  8,  1],
        [ 8,  1, 15],
        [ 1, 15, 10],
        [15, 10, 13],
        [10, 13, 17],
        [13, 17, 17],
        [17, 17,  5],
        [ 0,  0,  0],
        [ 0,  0, 11],
        [ 0, 11,  9],
        [1

In [80]:
Y

tensor([ 5, 11, 11,  1,  0, 13, 10,  9, 19,  9,  1,  0,  1, 19,  1,  0,  9, 16,
         1,  2,  5, 10, 10,  1,  0, 16, 13, 14,  8,  9,  1,  0,  3,  8,  1, 15,
        10, 13, 17, 17,  5,  0, 11,  9,  1,  0,  1, 11,  5, 10,  9,  1,  0,  8,
         1, 15, 14,  5, 15,  0,  5, 19,  5, 10, 20, 12,  0,  1,  2,  9,  7,  1,
         9, 10,  0,  5, 11,  9, 10, 20,  0,  5, 10,  9, 21,  1,  2,  5, 17,  8,
         0, 11,  9, 10,  1,  0,  5, 10, 10,  1,  0,  1, 19,  5, 15, 20,  0, 16,
        13,  6,  9,  1,  0,  3,  1, 11,  9, 10,  1,  0,  1, 15,  9,  1,  0, 16,
         3,  1, 15, 10,  5, 17, 17,  0, 19,  9,  3, 17, 13, 15,  9,  1,  0, 11,
         1,  4,  9, 16, 13, 12,  0, 10, 18, 12,  1,  0,  7, 15,  1,  3,  5,  0,
         3,  8, 10, 13,  5,  0, 14,  5, 12,  5, 10, 13, 14,  5,  0, 10,  1, 20,
        10,  1,  0, 15,  9, 10,  5, 20,  0, 21, 13,  5, 20,  0, 12, 13, 15,  1,
         0])

In [ ]:
# 4. Modèle MLP

class MakemoreMLP(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=10, n_hidden=200):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.fc1 = nn.Linear(block_size * n_embd, n_hidden)
        self.fc2 = nn.Linear(n_hidden, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        x = emb.view(emb.shape[0], -1)
        x = torch.tanh(self.fc1(x))
        logits = self.fc2(x)
        return logits

model = MakemoreMLP(vocab_size, block_size)

In [82]:
# 5. Entraînement

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(20000):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 2000 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

step 0: loss = 3.1049
step 2000: loss = 0.5504
step 4000: loss = 0.5505
step 6000: loss = 0.5503
step 8000: loss = 0.5508
step 10000: loss = 0.5503
step 12000: loss = 0.5503
step 14000: loss = 0.5504
step 16000: loss = 0.5504
step 18000: loss = 0.5502


In [83]:
# 6. Génération

@torch.no_grad()
def generate(model, max_length=20):
    out = []
    context = [0] * block_size

    for _ in range(max_length):
        x = torch.tensor([context])
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()

        if ix == 0:
            break

        out.append(itos[ix])
        context = context[1:] + [ix]

    return ''.join(out)

# 7. Test

for _ in range(20):
    print(generate(model))

luna
avery
avery
zoey
madison
riley
layla
emila
riley
luna
avery
olivia
olivia
grace
amelia
avery
madison
zoey
chloe
emma


In [ ]:
# Version Bigram simple

def train_bigram(words):
    N = torch.zeros((vocab_size, vocab_size))

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            i = stoi[ch1]
            j = stoi[ch2]
            N[i, j] += 1

    P = (N + 1)
    P /= P.sum(1, keepdim=True)
    return P

P = train_bigram(words)

@torch.no_grad()
def generate_bigram(P):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1).item()
        if ix == 0:
            break
        out.append(itos[ix])
    return ''.join(out)

for _ in range(10):
    print(generate_bigram(P))



=== BIGRAM ===
y
zuerlilzonupeohsyoea
saccahmicoyssarahotty
lema
asonhhmf
iauo
etsmhruruhle
aotvha
elfiammnrloeifhoha
nymlhfcdgrihy
